# Lab: RAG Basics - Building Retrieval Pipeline

## Learning Objectives

By the end of this lab, you will be able to:
- Understand the core components of a retrieval pipeline
- Implement document chunking strategies
- Create and work with text embeddings
- Build a vector-based retrieval system
- Evaluate retrieval quality

## Introduction

Retrieval-Augmented Generation (RAG) is a technique that enhances language models by retrieving relevant information from a knowledge base before generating responses. The retrieval pipeline is the foundation of any RAG system, responsible for finding the most relevant information given a query.

In this lab, you will build a complete retrieval pipeline from scratch, learning each component step by step.

## Setup

### Required Libraries

Install the following libraries:

```bash
pip install sentence-transformers numpy scikit-learn
```
**Installed the required libraries**

### Sample Dataset

For this lab, we'll work with a collection of documents about artificial intelligence concepts. Create a file called `documents.py` with the following content:

```python
DOCUMENTS = [
    "Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed.",
    "Deep learning is a type of machine learning based on artificial neural networks with multiple layers, allowing computers to learn complex patterns.",
    "Natural language processing (NLP) is a field of AI that focuses on the interaction between computers and human language, enabling machines to understand and generate text.",
    "Computer vision is an AI field that trains computers to interpret and understand visual information from the world, such as images and videos.",
    "Reinforcement learning is a machine learning approach where an agent learns to make decisions by performing actions and receiving rewards or penalties.",
    "Transfer learning is a technique where a model developed for one task is reused as the starting point for a model on a second task.",
    "Neural networks are computing systems inspired by biological neural networks, consisting of interconnected nodes that process information.",
    "Supervised learning involves training a model on labeled data, where the correct output is known for each input example.",
    "Unsupervised learning involves training models on data without labeled responses, finding hidden patterns or structures in the data.",
    "Transformers are a type of neural network architecture that uses self-attention mechanisms, revolutionary for NLP tasks."
]
```

In [1]:
# documents.py
# This cell allows us to do tasks without the need to import documents.py from somewhere else

DOCUMENTS = [
    "Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed.",
    "Deep learning is a type of machine learning based on artificial neural networks with multiple layers, allowing computers to learn complex patterns.",
    "Natural language processing (NLP) is a field of AI that focuses on the interaction between computers and human language, enabling machines to understand and generate text.",
    "Computer vision is an AI field that trains computers to interpret and understand visual information from the world, such as images and videos.",
    "Reinforcement learning is a machine learning approach where an agent learns to make decisions by performing actions and receiving rewards or penalties.",
    "Transfer learning is a technique where a model developed for one task is reused as the starting point for a model on a second task.",
    "Neural networks are computing systems inspired by biological neural networks, consisting of interconnected nodes that process information.",
    "Supervised learning involves training a model on labeled data, where the correct output is known for each input example.",
    "Unsupervised learning involves training models on data without labeled responses, finding hidden patterns or structures in the data.",
    "Transformers are a type of neural network architecture that uses self-attention mechanisms, revolutionary for NLP tasks."
]

## Part 1: Document Preprocessing and Chunking

### Understanding the Importance

Before we can retrieve documents, we need to prepare them properly. In real-world scenarios, documents are often too large to process as single units. Chunking breaks documents into smaller, manageable pieces while preserving context.

**Why this matters:** Poor chunking can lead to lost context or irrelevant retrievals, directly impacting the quality of your RAG system.

## Part 1: Document Preprocessing and Chunking

### Understanding the Importance

Before we can retrieve documents, we need to prepare them properly. In real-world scenarios, documents are often too large to process as single units. Chunking breaks documents into smaller, manageable pieces while preserving context.

**Why this matters:** Poor chunking can lead to lost context or irrelevant retrievals, directly impacting the quality of your RAG system.

### Task 1.1: Implement Basic Chunking

Create a function that splits documents into chunks based on a maximum token/word count.

```python
def chunk_by_words(text, max_words=50, overlap=10):
    """
    Split text into chunks based on word count.
    
    Args:
        text: Input text to chunk
        max_words: Maximum words per chunk
        overlap: Number of overlapping words between chunks
    
    Returns:
        List of text chunks
    """
    # Your code here
    pass
```

**Test your implementation:**

```python
sample_text = "Machine learning is a subset of artificial intelligence. It enables systems to learn from experience. Deep learning uses neural networks with multiple layers."

chunks = chunk_by_words(sample_text, max_words=10, overlap=3)
print(f"Number of chunks: {len(chunks)}")
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}: {chunk}")
```


**Explanation**

**Task 1.1:Word-based document chunking**

In this task, I implemented a function to split a large text into smaller chunks based on a fixed number of words. Chunking is necessary in RAG systems because large documents are difficult to process and retrieve effectively as a whole.

I used a sliding window approach, where each chunk contains a maximum number of words (max_words) and shares some overlapping words (overlap) with the next chunk. The overlap helps maintain context between chunks so that important information is not lost at the boundaries.

The step size is calculated as:

step = max_words - overlap

This ensures smooth transitions between chunks while covering the entire text.

The overall time complexity is O(n) since the text is processed once, and the space complexity is also O(n) because we store the generated chunks.

In [2]:
# Solution for Task 1.1

def chunk_by_words(text, max_words=50, overlap=10):
    """
    Split text into chunks based on word count.
    
    Args:
        text (str): Input text to chunk
        max_words (int): Maximum words allowed per chunk
        overlap (int): Number of overlapping words between consecutive chunks
    
    Returns:
        list: List of text chunks
    """

    # ==========================
    #  Input Validation
    # ==========================
    if max_words <= 0:
        raise ValueError("max_words must be greater than 0")

    if overlap >= max_words:
        raise ValueError("overlap must be smaller than max_words")

    # ==========================
    #  Convert Text → Words
    # ==========================
    # Splitting text by whitespace to get word list
    words = text.split()

    chunks = []

    # Step size defines how much we move forward
    # Example:
    # if max_words = 10 and overlap = 3
    # step = 7 (because 3 words overlap with next chunk)
    step = max_words - overlap

    # ==========================
    #  Sliding Window Logic
    # ==========================
    for start_index in range(0, len(words), step):

        # Select words from start_index to start_index + max_words
        chunk_words = words[start_index:start_index + max_words]

        # If chunk is empty, stop
        if not chunk_words:
            break

        # Join words back into sentence format
        chunk_text = " ".join(chunk_words)

        chunks.append(chunk_text)

        # If we reached the end, break early
        if start_index + max_words >= len(words):
            break

    return chunks

In [3]:
# Testing implementation

sample_text = "Machine learning is a subset of artificial intelligence. It enables systems to learn from experience. Deep learning uses neural networks with multiple layers."

chunks = chunk_by_words(sample_text, max_words=10, overlap=3)

print(f"Number of chunks: {len(chunks)}")
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}: {chunk}")

Number of chunks: 3
Chunk 1: Machine learning is a subset of artificial intelligence. It enables
Chunk 2: intelligence. It enables systems to learn from experience. Deep learning
Chunk 3: experience. Deep learning uses neural networks with multiple layers.


**Observation**

The function produced 3 chunks using the specified max_words and overlap values.The output shows that each chunk contains up to 10 words, and consecutive chunks share overlapping words such as “intelligence. It enables” and “experience. Deep learning”. 

This confirms that the sliding window approach is working correctly.The overlap preserves contextual continuity between chunks, preventing loss of meaning at chunk boundaries, which is important for improving retrieval quality in RAG systems.

### Task 1.2: Sentence-Based Chunking

Implement a more sophisticated chunking strategy that respects sentence boundaries.

```python
def chunk_by_sentences(text, max_sentences=2):
    """
    Split text into chunks based on complete sentences.
    
    Args:
        text: Input text to chunk
        max_sentences: Maximum sentences per chunk
    
    Returns:
        List of text chunks
    """
    # Your code here
    # Hint: Use simple sentence splitting on periods followed by spaces
    pass
```

**Questions to consider:**
- What are the advantages of sentence-based chunking over word-based chunking?
- When might word-based chunking be more appropriate?

**Explanation**

**Task 1.2 – Sentence-Based Document Chunking**

In this task, I implemented a chunking strategy that splits text based on complete sentences instead of a fixed number of words. This approach is more semantically meaningful because it respects natural language boundaries.

First, the text is split into individual sentences using a simple rule (splitting on ". "). Then, groups of sentences are combined together based on the max_sentences limit to form chunks.

Compared to word-based chunking, sentence-based chunking preserves complete thoughts and reduces the risk of breaking important context in the middle of a sentence. This can improve retrieval quality in RAG systems.

In [4]:
# Solution to Task 1.2

def chunk_by_sentences(text, max_sentences=2):
    """
    Split text into chunks based on complete sentences.
    
    Args:
        text (str): Input text to chunk
        max_sentences (int): Maximum sentences per chunk
    
    Returns:
        list: List of text chunks
    """

    if max_sentences <= 0:
        raise ValueError("max_sentences must be greater than 0")

    # ---------------------------------------
    #  Split text into sentences
    # ---------------------------------------
    # Using simple splitting rule as suggested:
    # Split on period followed by space
    sentences = text.split(". ")

    # Add the period back (except possibly the last one)
    sentences = [s if s.endswith(".") else s + "." for s in sentences]

    chunks = []

    # ---------------------------------------
    #  Group sentences into chunks
    # ---------------------------------------
    for i in range(0, len(sentences), max_sentences):

        # Select a group of sentences
        chunk_sentences = sentences[i:i + max_sentences]

        # Join them back into a single text chunk
        chunk_text = " ".join(chunk_sentences).strip()

        chunks.append(chunk_text)

    return chunks

In [5]:
# Testing Implementation

sample_text = (
    "Machine learning is a subset of artificial intelligence. "
    "It enables systems to learn from experience. "
    "Deep learning uses neural networks with multiple layers. "
    "Natural language processing focuses on language understanding."
)

chunks = chunk_by_sentences(sample_text, max_sentences=2)

print("Number of chunks:", len(chunks))
print()

for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}:")
    print(chunk)
    print()

Number of chunks: 2

Chunk 1:
Machine learning is a subset of artificial intelligence. It enables systems to learn from experience.

Chunk 2:
Deep learning uses neural networks with multiple layers. Natural language processing focuses on language understanding.



**Observation**


The function generated 2 chunks, with each chunk containing up to 2 complete sentences as specified. The output clearly preserves sentence boundaries,and no sentence is broken in the middle.

Chunk 1 contains two logically connected sentences about machine learning,while Chunk 2 groups two separate but related AI concepts. This confirms that the sentence-based chunking approach maintains semantic completeness and produces meaningful text segments suitable for embedding generation in a RAG system.

**Questions considered**

**What are the advantages of sentence-based chunking over word-based chunking?**

1.Preserves complete thoughts and ideas.

2.Avoids cutting sentences in the middle.

3.Produces more meaningful embeddings.

4.Often improves retrieval quality in RAG systems.

**When might word-based chunking be more appropriate?**

1.When strict token limits must be maintained.

2.When working with very long documents.

3.When more controlled chunk sizes are required for embedding models.

4.When overlapping context is needed for fine-grained retrieval.    

## Part 2: Creating Embeddings

### Understanding the Importance

Embeddings convert text into numerical vectors that capture semantic meaning. This transformation is crucial because computers can't directly compare text semantically, but they can measure distances between vectors.

**Why this matters:** The quality of embeddings directly determines how well your retrieval system understands semantic similarity. Poor embeddings will retrieve irrelevant documents even if your retrieval logic is perfect.


### Task 2.1: Initialize Embedding Model

Load a pre-trained sentence transformer model.

```python
from sentence_transformers import SentenceTransformer

def initialize_embedding_model(model_name='all-MiniLM-L6-v2'):
    """
    Initialize a sentence transformer model for creating embeddings.
    
    Args:
        model_name: Name of the pre-trained model
    
    Returns:
        Loaded model
    """
    # Your code here
    pass
```

**Explanation**

**Task 2.1 – Initializing the Embedding Model**

In this task, I created a function to initialize a pre-trained Sentence Transformer model. Embedding models are essential in a RAG pipeline because they convert text into dense numerical vectors that capture semantic meaning.

I used the SentenceTransformer class from the sentence-transformers library. The default model selected is 'all-MiniLM-L6-v2', which is lightweight and efficient while still providing good semantic representation.

By placing the model initialization inside a function, the code becomes modular and reusable. This also allows flexibility to change the model name if needed.

In [6]:
# Solution for Task 2.1

from sentence_transformers import SentenceTransformer

def initialize_embedding_model(model_name='all-MiniLM-L6-v2'):
    """
    Initialize a sentence transformer model for creating embeddings.
    
    Args:
        model_name (str): Name of the pre-trained model
    
    Returns:
        SentenceTransformer: Loaded embedding model
    """

    if not isinstance(model_name, str) or len(model_name.strip()) == 0:
        raise ValueError("model_name must be a valid non-empty string")

    # Load the specified pre-trained model
    model = SentenceTransformer(model_name)

    return model

In [7]:
# Testing Implementation 

model = initialize_embedding_model()

print("=== Embedding Model Initialization ===")

print(f"Model Architecture     : BERT-based Transformer")
print(f"Max Sequence Length    : {model[0].max_seq_length}")
print(f"Embedding Dimension    : {model[1].word_embedding_dimension}")

print("=" * 45)
print("Status: Model loaded successfully.")
print("=" * 45)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


=== Embedding Model Initialization ===
Model Architecture     : BERT-based Transformer
Max Sequence Length    : 256
Embedding Dimension    : 384
Status: Model loaded successfully.


**Observation**

The embedding model loaded successfully, as confirmed by the printed SentenceTransformer architecture. The model consists of a transformer(BERT-based backbone), a pooling layer, and a normalization layer,with an embedding dimension of 384.

**The warnings regarding unauthenticated HF requests and the
"UNEXPECTED" key (embeddings.position_ids) are informational and do not affect model functionality. These are common when loading pre-trained models and can be safely ignored.**

The model is now properly initialized and ready for embedding generation.

### Task 2.2: Generate Document Embeddings

Create embeddings for all documents in your knowledge base.

```python
def create_embeddings(documents, model):
    """
    Generate embeddings for a list of documents.
    
    Args:
        documents: List of text documents
        model: Embedding model
    
    Returns:
        numpy array of embeddings
    """
    # Your code here
    pass
```

**Test your implementation:**

```python
from documents import DOCUMENTS

model = initialize_embedding_model()
embeddings = create_embeddings(DOCUMENTS, model)

print(f"Number of documents: {len(DOCUMENTS)}")
print(f"Embedding shape: {embeddings.shape}")
print(f"First embedding (first 10 dimensions): {embeddings[0][:10]}")
```

**Explanation**

**Task 2.2 – Generating Document Embeddings**

In this task, I implemented a function to generate vector embeddings for all documents in the knowledge base. Embeddings convert text into dense numerical representations that capture semantic meaning, which is essential for similarity search in a RAG pipeline.

The function takes a list of documents and a pre-initialized Sentence Transformer model. It uses the model’s encode() method to transform each document into a fixed-dimensional vector. The output is returned as a NumPy array for efficient mathematical operations during retrieval.

This step is important because these embeddings will later be used to compute similarity scores between user queries and stored documents.

In [8]:
# Solution for Tsak 2.2

import numpy as np

def create_embeddings(documents, model):
    """
    Generate embeddings for a list of documents.
    
    Args:
        documents (list): List of text documents
        model (SentenceTransformer): Embedding model
    
    Returns:
        numpy.ndarray: Array of document embeddings
    """

    if not isinstance(documents, list):
        raise ValueError("documents must be a list of text strings")

    if len(documents) == 0:
        raise ValueError("documents list cannot be empty")

    # Generate embeddings using the model
    # convert_to_numpy=True ensures we get a NumPy array directly
    embeddings = model.encode(documents, convert_to_numpy=True)

    return embeddings

In [9]:
# Testing Implementation

model = initialize_embedding_model()
embeddings = create_embeddings(DOCUMENTS, model)

print(f"Number of documents: {len(DOCUMENTS)}")
print(f"Embedding shape: {embeddings.shape}")
print(f"First embedding (first 10 dimensions): {embeddings[0][:10]}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Number of documents: 10
Embedding shape: (10, 384)
First embedding (first 10 dimensions): [ 0.00840923 -0.00360582  0.05522148  0.06061028  0.03103588 -0.02438604
 -0.02974018 -0.03058413 -0.07072204 -0.00281697]


**Observation**

**The warning is an informational warning related to internal transformer model components and does not affect embedding generation or retrieval performance.The model loads successfully and functions correctly despite this message.**

The number of generated embeddings matches the number of input documents (10),confirming that each document has been successfully converted into a vector representation.

The embedding shape is (10, 384), which indicates that there are 10 document vectors,each with 384 dimensions. This matches the expected embedding size of the'all-MiniLM-L6-v2' model.

The output confirms that the embedding generation step is functioning correctly and that the document vectors are now ready to be used for similarity comparison and retrieval in the RAG pipeline.

### Task 2.3: Understanding Embedding Space

Calculate and visualize the similarity between different documents.

```python
import numpy as np

def cosine_similarity(vec1, vec2):
    """
    Calculate cosine similarity between two vectors.
    
    Args:
        vec1: First vector
        vec2: Second vector
    
    Returns:
        Similarity score between -1 and 1
    """
    # Your code here
    pass
```

**Experiment:**

```python
# Compare similarities between different document pairs
# Document 0: Machine learning definition
# Document 1: Deep learning definition
# Document 4: Reinforcement learning definition

similarity_ml_dl = cosine_similarity(embeddings[0], embeddings[1])
similarity_ml_rl = cosine_similarity(embeddings[0], embeddings[4])

print(f"Similarity (ML vs DL): {similarity_ml_dl:.4f}")
print(f"Similarity (ML vs RL): {similarity_ml_rl:.4f}")
```

**Questions:**
- Which pair is more similar? Why does this make sense?
- What does the similarity score tell you about semantic relationships?

**Explanation**

**Task 2.3 – Understanding Embedding Space Using Cosine Similarity**

In this task, I implemented cosine similarity to measure how semantically similar two document embeddings are. Since embeddings are numerical vector representations of text, we need a mathematical method to compare them.

Cosine similarity measures the angle between two vectors. If the vectors point in similar directions, their cosine similarity will be close to 1, indicating strong semantic similarity. If they are unrelated, the value will be closer to 0.

This is important in a RAG pipeline because similarity scores help retrieve the most relevant documents for a given query.

In [10]:
# Solution for Task 2.3

import numpy as np

def cosine_similarity(vec1, vec2):
    """
    Calculate cosine similarity between two vectors.
    
    Args:
        vec1 (numpy.ndarray): First embedding vector
        vec2 (numpy.ndarray): Second embedding vector
    
    Returns:
        float: Similarity score between -1 and 1
    """

    # Convert inputs to numpy arrays (safety check)
    vec1 = np.array(vec1)
    vec2 = np.array(vec2)

    # Compute dot product between vectors
    dot_product = np.dot(vec1, vec2)

    # Compute magnitudes (L2 norms)
    norm_vec1 = np.linalg.norm(vec1)
    norm_vec2 = np.linalg.norm(vec2)

    # Avoid division by zero
    if norm_vec1 == 0 or norm_vec2 == 0:
        raise ValueError("One of the vectors has zero magnitude.")

    # Apply cosine similarity formula
    similarity = dot_product / (norm_vec1 * norm_vec2)

    return similarity

In [11]:
# Testing Implementation

# Compare similarities between different document pairs
# Document 0: Machine learning definition
# Document 1: Deep learning definition
# Document 4: Reinforcement learning definition

similarity_ml_dl = cosine_similarity(embeddings[0], embeddings[1])
similarity_ml_rl = cosine_similarity(embeddings[0], embeddings[4])

print(f"Similarity (ML vs DL): {similarity_ml_dl:.4f}")
print(f"Similarity (ML vs RL): {similarity_ml_rl:.4f}")

Similarity (ML vs DL): 0.5919
Similarity (ML vs RL): 0.6325


**Observation**

The similarity score between Machine Learning and Reinforcement Learning (0.6325) is slightly higher than the similarity between Machine Learning and Deep Learning(0.5919).

This indicates that, in this embedding space, reinforcement learning is represented as more closely related to general machine learning. Since reinforcement learning is a type of machine learning approach, this semantic closeness is reasonable.

The cosine similarity values confirm that the embedding model captures meaning-based relationships rather than exact word overlap.

**Questions:**

**Which pair is more similar? Why does this make sense?**

The pair Machine Learning and Reinforcement Learning (0.6325) is more similar than Machine Learning and Deep Learning (0.5919).

This makes sense because reinforcement learning is explicitly described as a type of machine learning approach in the document. The shared terminology and similar explanation style increase the semantic overlap between the two texts, resulting in a higher similarity score.
  
**What does the similarity score tell you about semantic relationships?**

The similarity score indicates how closely two texts are related in meaning.A value closer to 1 means the texts are semantically similar, while a value closer to 0 suggests weaker semantic connection.

In this case, both scores are above 0.5, which shows that all compared documents are conceptually related within the broader field of AI.This demonstrates that embeddings capture semantic relationships rather than relying on exact keyword matching. 

## Part 3: Building the Retrieval System

### Understanding the Importance

The retrieval system is the core of your pipeline. It takes a query and finds the most relevant documents from your knowledge base. The efficiency and accuracy of this component directly impact the performance of your entire RAG system.

**Why this matters:** A slow or inaccurate retrieval system will create bottlenecks and provide poor context to your language model, resulting in irrelevant or incorrect responses.

### Task 3.1: Implement Basic Retrieval

Create a function that retrieves the top-k most similar documents for a given query.

```python
def retrieve_documents(query, documents, embeddings, model, top_k=3):
    """
    Retrieve the most relevant documents for a query.
    
    Args:
        query: Search query string
        documents: List of document texts
        embeddings: Pre-computed document embeddings
        model: Embedding model
        top_k: Number of documents to retrieve
    
    Returns:
        List of tuples (document, similarity_score)
    """
    # Your code here
    # Steps:
    # 1. Create embedding for the query
    # 2. Calculate similarity between query and all documents
    # 3. Sort by similarity and return top_k results
    pass
```

**Test your retrieval system:**

```python
query = "How do computers learn from data?"
results = retrieve_documents(query, DOCUMENTS, embeddings, model, top_k=3)

print(f"Query: {query}\n")
for i, (doc, score) in enumerate(results, 1):
    print(f"Result {i} (Score: {score:.4f}):")
    print(f"{doc}\n")
```

**Explanation**

**Task 3.1 – Implementing Basic Retrieval**

In this task, I implemented a retrieval function that identifies the top-k most relevant documents for a given query.

The retrieval process consists of three main steps:
1. Convert the query into an embedding using the same embedding model used for the documents.
2. Compute cosine similarity between the query embedding and all document embeddings.
3. Sort the documents by similarity score and return the top_k results.

This approach ensures semantic retrieval, meaning that documents are selected based on meaning rather than exact keyword matching.This is a fundamental component of a Retrieval-Augmented Generation (RAG) system.

In [12]:
# Solution to Task 3.1

import numpy as np

def retrieve_documents(query, documents, embeddings, model, top_k=3):
    """
    Retrieve the most relevant documents for a query.
    
    Args:
        query (str): Search query string
        documents (list): List of document texts
        embeddings (numpy.ndarray): Pre-computed document embeddings
        model (SentenceTransformer): Embedding model
        top_k (int): Number of documents to retrieve
    
    Returns:
        list: List of tuples (document, similarity_score)
    """

    if not isinstance(query, str) or len(query.strip()) == 0:
        raise ValueError("Query must be a non-empty string.")

    if top_k <= 0:
        raise ValueError("top_k must be greater than 0.")

    # Step 1: Generate embedding for the query
    query_embedding = model.encode(query, convert_to_numpy=True)

    # Step 2: Compute cosine similarity with all document embeddings
    similarities = []
    for i, doc_embedding in enumerate(embeddings):
        score = cosine_similarity(query_embedding, doc_embedding)
        similarities.append((documents[i], score))

    # Step 3: Sort by similarity score (highest first)
    similarities.sort(key=lambda x: x[1], reverse=True)

    # Return top_k results
    return similarities[:top_k]

In [13]:
# Testing Implementation

query = "How do computers learn from data?"
results = retrieve_documents(query, DOCUMENTS, embeddings, model, top_k=3)

print(f"Query: {query}\n")
for i, (doc, score) in enumerate(results, 1):
    print(f"Result {i} (Score: {score:.4f}):")
    print(f"{doc}\n")

Query: How do computers learn from data?

Result 1 (Score: 0.5245):
Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed.

Result 2 (Score: 0.5062):
Supervised learning involves training a model on labeled data, where the correct output is known for each input example.

Result 3 (Score: 0.4734):
Unsupervised learning involves training models on data without labeled responses, finding hidden patterns or structures in the data.



**Observation**

For the query "How do computers learn from data?", the retrieval system
correctly identified Machine Learning as the most relevant document
with the highest similarity score (0.5245). This is appropriate because
machine learning directly explains how systems learn from experience.

The second and third results correspond to Supervised Learning (0.5062)
and Unsupervised Learning (0.4734), both of which are specific learning
approaches that involve training models on data. Their high similarity
scores indicate strong semantic relevance to the query.

The decreasing similarity scores reflect a logical ranking of related
concepts, confirming that the retrieval function successfully captures
semantic meaning and ranks documents appropriately within the RAG pipeline.

### Task 3.2: Implement Retrieval with Threshold

Enhance your retrieval function to only return documents above a similarity threshold.

```python
def retrieve_with_threshold(query, documents, embeddings, model, 
                           top_k=3, threshold=0.3):
    """
    Retrieve documents that meet a minimum similarity threshold.
    
    Args:
        query: Search query string
        documents: List of document texts
        embeddings: Pre-computed document embeddings
        model: Embedding model
        top_k: Maximum number of documents to retrieve
        threshold: Minimum similarity score
    
    Returns:
        List of tuples (document, similarity_score)
    """
    # Your code here
    pass
```

**Experiment with different thresholds:**

```python
queries = [
    "What is deep learning?",
    "How do neural networks work?",
    "Tell me about quantum computing"  # Out of domain query
]

for query in queries:
    print(f"\nQuery: {query}")
    results = retrieve_with_threshold(query, DOCUMENTS, embeddings, model, 
                                     top_k=3, threshold=0.3)
    print(f"Retrieved {len(results)} documents")
    if results:
        print(f"Top result (Score: {results[0][1]:.4f}): {results[0][0][:100]}...")
```

**Questions:**
- What happens when you query about topics not in your knowledge base?
- How does the threshold help filter irrelevant results?

**Explanation**

**Task 3.2 – Retrieval with Similarity Threshold**

In this task, I enhanced the retrieval function by introducing a similarity threshold.
Instead of always returning the top_k documents, the function now filters out
documents whose similarity score is below a specified threshold.

This improves retrieval quality by preventing weakly related or irrelevant
documents from being returned. The threshold ensures that only sufficiently
relevant results are included in the final output.

This mechanism is especially important in RAG systems to avoid passing
irrelevant context to the language model.

In [14]:
# Solution for Task 3.2

def retrieve_with_threshold(query, documents, embeddings, model, 
                            top_k=3, threshold=0.3):
    """
    Retrieve documents that meet a minimum similarity threshold.
    
    Args:
        query (str): Search query string
        documents (list): List of document texts
        embeddings (numpy.ndarray): Pre-computed document embeddings
        model (SentenceTransformer): Embedding model
        top_k (int): Maximum number of documents to retrieve
        threshold (float): Minimum similarity score
    
    Returns:
        list: List of tuples (document, similarity_score)
    """

    if not isinstance(query, str) or len(query.strip()) == 0:
        raise ValueError("Query must be a non-empty string.")

    if top_k <= 0:
        raise ValueError("top_k must be greater than 0.")

    # Step 1: Generate embedding for query
    query_embedding = model.encode(query, convert_to_numpy=True)

    # Step 2: Calculate similarities
    results = []
    for i, doc_embedding in enumerate(embeddings):
        score = cosine_similarity(query_embedding, doc_embedding)

        # Step 3: Apply threshold filter
        if score >= threshold:
            results.append((documents[i], score))

    # Step 4: Sort filtered results by score (descending)
    results.sort(key=lambda x: x[1], reverse=True)

    # Step 5: Return up to top_k results
    return results[:top_k]

In [15]:
# Testing Implementation

# Experimenting with different thresholds
queries = [
    "What is deep learning?",
    "How do neural networks work?",
    "Tell me about quantum computing"  # Out of domain query
]

for query in queries:
    print(f"\nQuery: {query}")
    results = retrieve_with_threshold(query, DOCUMENTS, embeddings, model, 
                                      top_k=3, threshold=0.3)
    print(f"Retrieved {len(results)} documents")
    if results:
        print(f"Top result (Score: {results[0][1]:.4f}): {results[0][0][:100]}...")


Query: What is deep learning?
Retrieved 3 documents
Top result (Score: 0.7955): Deep learning is a type of machine learning based on artificial neural networks with multiple layers...

Query: How do neural networks work?
Retrieved 3 documents
Top result (Score: 0.7190): Neural networks are computing systems inspired by biological neural networks, consisting of intercon...

Query: Tell me about quantum computing
Retrieved 1 documents
Top result (Score: 0.3105): Neural networks are computing systems inspired by biological neural networks, consisting of intercon...


**Observation**

For the query "What is deep learning?", the system retrieved 3 documents,
with the highest similarity score (0.7955) correctly corresponding to the
Deep Learning definition. This shows strong semantic matching.

For "How do neural networks work?", the top result (0.7190) was the
Neural Networks definition, which is highly relevant to the query.
This confirms that the retrieval system correctly identifies closely
related concepts.

For the out-of-domain query "Tell me about quantum computing",
only 1 document was retrieved with a relatively low similarity score
(0.3105). This indicates that the system could not find strongly relevant
matches in the knowledge base, and only a weakly related document
(neural networks) passed the threshold.

Overall, the threshold effectively reduces irrelevant results while still
allowing partially related content when similarity exceeds the minimum limit.

**Answers**

**What happens when you query about topics not in your knowledge base?**

When querying an out-of-domain topic such as quantum computing,
the system retrieves very few results and the similarity scores are low.
This indicates that the model cannot find strong semantic matches
within the existing documents.

This behavior is expected because the knowledge base does not
contain information about quantum computing.


**How does the threshold help filter irrelevant results?**

The threshold ensures that only documents with similarity scores
above a specified minimum value are returned. This prevents
very weakly related documents from being included in the results.

In this case, the threshold reduced the number of returned documents for the unrelated query, helping control noise in the retrieval process.This improves the reliability of the RAG system.

## Part 4: Optimizing the Retrieval Pipeline

### Understanding the Importance

As your knowledge base grows, retrieval performance becomes critical. You need efficient data structures and algorithms to handle thousands or millions of documents.

**Why this matters:** A retrieval system that works well with 10 documents might become unusable with 10,000. Learning optimization techniques early prevents scaling problems later.

### Task 4.1: Implement Batch Processing

Create a function that efficiently handles multiple queries at once.

```python
def batch_retrieve(queries, documents, embeddings, model, top_k=3):
    """
    Efficiently retrieve documents for multiple queries.
    
    Args:
        queries: List of query strings
        documents: List of document texts
        embeddings: Pre-computed document embeddings
        model: Embedding model
        top_k: Number of documents per query
    
    Returns:
        Dictionary mapping each query to its results
    """
    # Your code here
    # Hint: Create embeddings for all queries at once
    pass
```


**Explanation**

**Task 4.1 – Batch Retrieval for Multiple Queries**

In this task, I implemented a batch retrieval function that processes
multiple queries efficiently. Instead of generating embeddings for each
query separately, the function generates embeddings for all queries in
a single batch.

Batch processing improves efficiency because embedding models are
optimized for vectorized computation. This reduces repeated model calls
and improves scalability when handling large numbers of queries.

After generating query embeddings, cosine similarity is computed
between each query and all document embeddings, and the top_k
results are returned for each query.

In [16]:
# Solution for Task 4.1

import numpy as np

def batch_retrieve(queries, documents, embeddings, model, top_k=3):
    """
    Efficiently retrieve documents for multiple queries.
    
    Args:
        queries (list): List of query strings
        documents (list): List of document texts
        embeddings (numpy.ndarray): Pre-computed document embeddings
        model (SentenceTransformer): Embedding model
        top_k (int): Number of documents per query
    
    Returns:
        dict: Dictionary mapping each query to its results
    """

    if not isinstance(queries, list) or len(queries) == 0:
        raise ValueError("queries must be a non-empty list")

    # Step 1: Generate embeddings for all queries at once (batch)
    query_embeddings = model.encode(queries, convert_to_numpy=True)

    results_dict = {}

    # Step 2: For each query embedding, compute similarities
    for q_idx, query_embedding in enumerate(query_embeddings):

        similarities = []

        for doc_idx, doc_embedding in enumerate(embeddings):
            score = cosine_similarity(query_embedding, doc_embedding)
            similarities.append((documents[doc_idx], score))

        # Step 3: Sort by similarity
        similarities.sort(key=lambda x: x[1], reverse=True)

        # Step 4: Store top_k results for that query
        results_dict[queries[q_idx]] = similarities[:top_k]

    return results_dict

In [17]:
# Testing Implementation

queries = [
    "What is supervised learning?",
    "Explain neural networks",
    "What is transfer learning?"
]

batch_results = batch_retrieve(queries, DOCUMENTS, embeddings, model, top_k=2)

for query, results in batch_results.items():
    print(f"\nQuery: {query}")
    for i, (doc, score) in enumerate(results, 1):
        print(f"Result {i} (Score: {score:.4f}): {doc[:100]}...")


Query: What is supervised learning?
Result 1 (Score: 0.8625): Supervised learning involves training a model on labeled data, where the correct output is known for...
Result 2 (Score: 0.5898): Unsupervised learning involves training models on data without labeled responses, finding hidden pat...

Query: Explain neural networks
Result 1 (Score: 0.7010): Neural networks are computing systems inspired by biological neural networks, consisting of intercon...
Result 2 (Score: 0.6182): Deep learning is a type of machine learning based on artificial neural networks with multiple layers...

Query: What is transfer learning?
Result 1 (Score: 0.8615): Transfer learning is a technique where a model developed for one task is reused as the starting poin...
Result 2 (Score: 0.4890): Supervised learning involves training a model on labeled data, where the correct output is known for...


**Observation**

The batch retrieval function successfully returned relevant results for
all queries while processing them efficiently in a single embedding step.

For "What is supervised learning?", the correct definition was retrieved
with a very high similarity score (0.8625), showing strong semantic alignment.
The second result (unsupervised learning) is also related, which indicates
that the model understands conceptual relationships between learning types.

For "Explain neural networks", the top result (0.7010) correctly identified
the neural network definition, followed by deep learning (0.6182), which is
closely connected to neural networks.

For "What is transfer learning?", the correct document was retrieved with
a high similarity score (0.8615), confirming accurate semantic matching.

Overall, the batch retrieval method produces logically ranked results while
efficiently handling multiple queries, demonstrating scalability and proper
semantic understanding within the retrieval pipeline.

### Task 4.2: Create a Retrieval Class

Encapsulate your retrieval pipeline in a reusable class.

```python
class RetrievalPipeline:
    """
    A complete retrieval pipeline for RAG systems.
    """
    
    def __init__(self, model_name='all-MiniLM-L6-v2'):
        """Initialize the pipeline with an embedding model."""
        # Your code here
        pass
    
    def add_documents(self, documents):
        """
        Add documents to the knowledge base.
        
        Args:
            documents: List of text documents
        """
        # Your code here
        pass
    
    def retrieve(self, query, top_k=3, threshold=0.0):
        """
        Retrieve relevant documents for a query.
        
        Args:
            query: Search query
            top_k: Number of results
            threshold: Minimum similarity score
        
        Returns:
            List of (document, score) tuples
        """
        # Your code here
        pass
    
    def get_stats(self):
        """Return statistics about the knowledge base."""
        # Your code here
        pass
```

**Test your class:**

```python
from documents import DOCUMENTS

pipeline = RetrievalPipeline()
pipeline.add_documents(DOCUMENTS)

print("Pipeline Statistics:")
print(pipeline.get_stats())

query = "What are neural networks?"
results = pipeline.retrieve(query, top_k=2)

print(f"\nQuery: {query}")
for doc, score in results:
    print(f"Score: {score:.4f} | {doc[:80]}...")
```

**Explanation**

**Task 4.2 – Creating a Retrieval Pipeline Class**

In this task, I encapsulated the entire retrieval workflow inside a class
called RetrievalPipeline. This improves modularity, reusability, and code
organization.

The class structure follows these responsibilities:

- __init__(): Initializes the embedding model and prepares storage.
- add_documents(): Stores documents and generates embeddings.
- retrieve(): Performs semantic retrieval with optional threshold filtering.
- get_stats(): Returns useful metadata about the knowledge base.

Encapsulating the logic into a class makes the system easier to scale,
extend, and reuse in real-world RAG applications.

In [18]:
# Solution for Task 4.2

class RetrievalPipeline:
    """
    A complete retrieval pipeline for RAG systems.
    """
    
    def __init__(self, model_name='all-MiniLM-L6-v2'):
        """
        Initialize the pipeline with an embedding model.
        """
        self.model = initialize_embedding_model(model_name)
        self.documents = []
        self.embeddings = None
    
    def add_documents(self, documents):
        """
        Add documents to the knowledge base.
        
        Args:
            documents (list): List of text documents
        """
        if not isinstance(documents, list) or len(documents) == 0:
            raise ValueError("documents must be a non-empty list.")
        
        self.documents = documents
        self.embeddings = create_embeddings(self.documents, self.model)
    
    def retrieve(self, query, top_k=3, threshold=0.0):
        """
        Retrieve relevant documents for a query.
        
        Args:
            query (str): Search query
            top_k (int): Number of results
            threshold (float): Minimum similarity score
        
        Returns:
            list: List of (document, score) tuples
        """
        if self.embeddings is None:
            raise ValueError("No documents added. Please call add_documents() first.")
        
        query_embedding = self.model.encode(query, convert_to_numpy=True)
        
        results = []
        for i, doc_embedding in enumerate(self.embeddings):
            score = cosine_similarity(query_embedding, doc_embedding)
            if score >= threshold:
                results.append((self.documents[i], score))
        
        results.sort(key=lambda x: x[1], reverse=True)
        return results[:top_k]
    
    def get_stats(self):
        """
        Return statistics about the knowledge base.
        """
        if self.embeddings is None:
            return {"num_documents": 0, "embedding_dimension": 0}
        
        return {
            "num_documents": len(self.documents),
            "embedding_dimension": self.embeddings.shape[1]
        }

In [19]:
# Testing Implementation

pipeline = RetrievalPipeline()
pipeline.add_documents(DOCUMENTS)

print("Pipeline Statistics:")
print(pipeline.get_stats())

query = "What are neural networks?"
results = pipeline.retrieve(query, top_k=2)

print(f"\nQuery: {query}")
for doc, score in results:
    print(f"Score: {score:.4f} | {doc[:80]}...")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Pipeline Statistics:
{'num_documents': 10, 'embedding_dimension': 384}

Query: What are neural networks?
Score: 0.7993 | Neural networks are computing systems inspired by biological neural networks, co...
Score: 0.6365 | Deep learning is a type of machine learning based on artificial neural networks ...


**Observation**

**This is an informational warning related to internal transformer model
components and does not affect embedding generation or retrieval performance.
The model loads successfully and functions correctly despite this message.**

The RetrievalPipeline class successfully initialized the embedding model
and generated embeddings for all 10 documents, with an embedding dimension
of 384 as expected for the selected model.

The get_stats() method correctly reports the number of documents and
embedding size, confirming that the knowledge base has been properly configured.

For the query "What are neural networks?", the system retrieved the correct
document with a high similarity score (0.7993), indicating strong semantic
alignment. The second result (deep learning) is also conceptually related,
which demonstrates that the ranking mechanism correctly captures meaningful
relationships between AI concepts.

Overall, the class-based design organizes the retrieval workflow effectively,
improves modularity, and provides a reusable structure suitable for scaling
RAG systems.

## Part 5: Evaluation and Analysis

### Understanding the Importance

You can't improve what you don't measure. Evaluation metrics help you understand how well your retrieval system performs and guide optimization efforts.

**Why this matters:** Without proper evaluation, you won't know if changes to your pipeline improve or degrade performance. In production systems, this can mean the difference between helpful and harmful AI responses.


### Task 5.1: Implement Relevance Metrics

Create functions to measure retrieval quality.

```python
def calculate_precision_at_k(retrieved_docs, relevant_docs, k=3):
    """
    Calculate precision at k: proportion of retrieved docs that are relevant.
    
    Args:
        retrieved_docs: List of retrieved document indices
        relevant_docs: Set of relevant document indices
        k: Number of top results to consider
    
    Returns:
        Precision score
    """
    # Your code here
    pass

def calculate_recall_at_k(retrieved_docs, relevant_docs, k=3):
    """
    Calculate recall at k: proportion of relevant docs that were retrieved.
    
    Args:
        retrieved_docs: List of retrieved document indices
        relevant_docs: Set of relevant document indices
        k: Number of top results to consider
    
    Returns:
        Recall score
    """
    # Your code here
    pass
```

**Explanation**

**Task 5.1 – Implementing Precision@K and Recall@K**

In this task, I implemented two important retrieval evaluation metrics:
Precision@K and Recall@K.

Precision@K measures the proportion of retrieved documents (within the
top k results) that are actually relevant.

Recall@K measures the proportion of all relevant documents that were
successfully retrieved within the top k results.

These metrics help evaluate the effectiveness of the retrieval system
by quantifying how accurately it identifies relevant documents.
Precision focuses on retrieval quality, while recall focuses on coverage

In [20]:
# Solution for Task 5.1

def calculate_precision_at_k(retrieved_docs, relevant_docs, k=3):
    """
    Calculate precision at k: proportion of retrieved docs that are relevant.
    
    Args:
        retrieved_docs (list): List of retrieved document indices
        relevant_docs (set): Set of relevant document indices
        k (int): Number of top results to consider
    
    Returns:
        float: Precision@k score
    """

    if k <= 0:
        raise ValueError("k must be greater than 0")

    # Consider only top k retrieved documents
    top_k_docs = retrieved_docs[:k]

    # Count how many of them are relevant
    relevant_retrieved = sum(1 for doc in top_k_docs if doc in relevant_docs)

    # Precision formula
    precision = relevant_retrieved / k

    return precision


def calculate_recall_at_k(retrieved_docs, relevant_docs, k=3):
    """
    Calculate recall at k: proportion of relevant docs that were retrieved.
    
    Args:
        retrieved_docs (list): List of retrieved document indices
        relevant_docs (set): Set of relevant document indices
        k (int): Number of top results to consider
    
    Returns:
        float: Recall@k score
    """

    if k <= 0:
        raise ValueError("k must be greater than 0")

    if len(relevant_docs) == 0:
        return 0.0

    # Consider only top k retrieved documents
    top_k_docs = retrieved_docs[:k]

    # Count how many relevant documents were retrieved
    relevant_retrieved = sum(1 for doc in top_k_docs if doc in relevant_docs)

    # Recall formula
    recall = relevant_retrieved / len(relevant_docs)

    return recall

In [21]:
# Testing Implementation

# Simulated retrieved results (indices)
retrieved_docs = [1, 0, 6]

# Define ground truth relevant documents
relevant_docs = {1}

precision = calculate_precision_at_k(retrieved_docs, relevant_docs, k=3)
recall = calculate_recall_at_k(retrieved_docs, relevant_docs, k=3)

print("Precision@3:", precision)
print("Recall@3:", recall)

Precision@3: 0.3333333333333333
Recall@3: 1.0


**Observation**

The Precision@3 value is 0.3333, meaning that only 1 out of the top 3
retrieved documents was actually relevant. This indicates that while
the system successfully included the correct document, two of the
retrieved documents were not relevant.

The Recall@3 value is 1.0, which means that all relevant documents
(in this case, one relevant document) were successfully retrieved
within the top 3 results.

This result demonstrates that the system achieved full coverage
(high recall) but moderate precision. In practical terms, the
retrieval system was able to find the correct document, but it also
included some irrelevant ones in the top results.

### Task 5.2: Create Test Cases

Define test queries with known relevant documents.

```python
# Test cases: (query, set of relevant document indices)
test_cases = [
    ("What is machine learning?", {0, 7, 8}),  # ML, supervised, unsupervised
    ("Explain neural networks", {1, 6, 9}),     # Deep learning, neural nets, transformers
    ("How does computer vision work?", {3}),    # Computer vision
]

def evaluate_retrieval(pipeline, test_cases, k=3):
    """
    Evaluate retrieval performance on test cases.
    
    Args:
        pipeline: RetrievalPipeline instance
        test_cases: List of (query, relevant_docs) tuples
        k: Number of results to retrieve
    
    Returns:
        Dictionary with average precision and recall
    """
    # Your code here
    # For each test case:
    # 1. Retrieve documents
    # 2. Get indices of retrieved documents
    # 3. Calculate precision and recall
    # 4. Average across all test cases
    pass
```

**Explanation**

**Task 5.2 – Evaluating Retrieval Using Test Cases**

In this task, I implemented a function to evaluate the overall
retrieval performance using multiple test queries with known
relevant documents.

For each test case:
1. The system retrieves the top-k documents.
2. The indices of the retrieved documents are identified.
3. Precision@k and Recall@k are calculated.
4. The scores are averaged across all test cases.

This provides a more comprehensive evaluation of the retrieval
system, rather than measuring performance on a single query.

In [22]:
# Solution for Task 5.2

def evaluate_retrieval(pipeline, test_cases, k=3):
    """
    Evaluate retrieval performance on test cases.
    
    Args:
        pipeline (RetrievalPipeline): RetrievalPipeline instance
        test_cases (list): List of (query, relevant_docs) tuples
        k (int): Number of results to retrieve
    
    Returns:
        dict: Dictionary with average precision and recall
    """

    total_precision = 0
    total_recall = 0
    num_tests = len(test_cases)

    for query, relevant_docs in test_cases:

        # Step 1: Retrieve documents
        results = pipeline.retrieve(query, top_k=k)

        # Step 2: Convert retrieved documents to indices
        retrieved_indices = [
            pipeline.documents.index(doc) for doc, _ in results
        ]

        # Step 3: Calculate metrics(precision and recall)
        precision = calculate_precision_at_k(
            retrieved_indices, relevant_docs, k=k
        )
        recall = calculate_recall_at_k(
            retrieved_indices, relevant_docs, k=k
        )

        total_precision += precision
        total_recall += recall

    # Step 4: Compute averages across all test cases
    avg_precision = total_precision / num_tests
    avg_recall = total_recall / num_tests

    return {
        "average_precision": avg_precision,
        "average_recall": avg_recall
    }

In [23]:
# Testing Implementation

# Define test cases
test_cases = [
    ("What is machine learning?", {0, 7, 8}),
    ("Explain neural networks", {1, 6, 9}),
    ("How does computer vision work?", {3}),
]

results = evaluate_retrieval(pipeline, test_cases, k=3)

print("=== Retrieval Evaluation Results ===")
print(f"Average Precision@3 : {results['average_precision']:.4f}")
print(f"Average Recall@3    : {results['average_recall']:.4f}")

=== Retrieval Evaluation Results ===
Average Precision@3 : 0.6667
Average Recall@3    : 0.8889


**Observation**

The retrieval system achieved an Average Precision@3 of 0.6667,
indicating that approximately two-thirds of the retrieved documents
across the test cases were relevant.

The Average Recall@3 is 0.8889, which shows that the system was able
to retrieve most of the relevant documents for the given queries.

This suggests that the retrieval pipeline has strong coverage (high recall)
but still includes some non-relevant documents in the top results,
leading to slightly lower precision.

Overall, the system demonstrates effective semantic retrieval performance,
with room for improvement in filtering less relevant results.

### Task 5.3: Analyze Retrieval Performance

Run evaluation and analyze the results.

```python
# Your code to evaluate and print results
```

**Analysis Questions:**
- What is the average precision and recall of your system?
- Which queries perform best/worst? Why?
- How could you improve retrieval quality?

**Explanation**

**Task 5.3 – Retrieval Performance Analysis**

In this task, I evaluated the overall performance of the retrieval
pipeline using multiple test queries with predefined relevant documents.

By computing average Precision@3 and Recall@3, I analyzed how accurately
the system retrieves relevant information and how well it covers the
expected relevant documents.

This analysis helps identify strengths and weaknesses of the current
retrieval strategy.

In [24]:
# Solution for Task 5.3

results = evaluate_retrieval(pipeline, test_cases, k=3)

print("=== Retrieval Performance Summary ===")
print(f"Average Precision@3 : {results['average_precision']:.4f}")
print(f"Average Recall@3    : {results['average_recall']:.4f}")

=== Retrieval Performance Summary ===
Average Precision@3 : 0.6667
Average Recall@3    : 0.8889


**Q1: What is the average precision and recall of your system?**

The system achieved an Average Precision@3 of 0.6667 and an
Average Recall@3 of 0.8889.

This means that most relevant documents are successfully retrieved
(high recall), but some non-relevant documents are still included
in the top results (moderate precision).


**Q2: Which queries perform best/worst? Why?**

Queries that match well-defined concepts such as
"How does computer vision work?" perform best because
there is a clear and direct document in the knowledge base.

Queries like "What is machine learning?" may retrieve
related concepts such as supervised and unsupervised learning,
which increases recall but slightly lowers precision due to
semantic overlap between related topics.


**Q3: How could you improve retrieval quality?**

Retrieval quality could be improved by:

- Increasing the similarity threshold to filter weaker matches.
- Using better chunking strategies to improve embedding quality.
- Applying re-ranking techniques after initial retrieval.
- Using larger or more advanced embedding models.
- Introducing metadata filtering or domain-specific indexing.

These improvements could increase precision while maintaining high recall.

## Part 6: Real-World Application

### Understanding the Importance

This final section bridges the gap between learning and application. You'll work with a more realistic scenario that includes challenges you'll face in production systems.

**Why this matters:** Textbook examples rarely capture the messiness of real data. Learning to handle longer documents, mixed content types, and edge cases prepares you for actual RAG implementations.

### Task 6.1: Extended Knowledge Base

Create a more complex dataset with longer documents.

```python
EXTENDED_DOCS = [
    """
    Artificial Intelligence (AI) is transforming industries worldwide. Machine learning, 
    a subset of AI, enables systems to learn from data without explicit programming. 
    Companies use ML for predictive analytics, recommendation systems, and automation. 
    The field has grown exponentially with the availability of big data and computational power.
    """,
    """
    Deep learning has revolutionized computer vision and natural language processing. 
    Convolutional neural networks (CNNs) excel at image recognition tasks, achieving 
    human-level performance in many cases. Applications include medical imaging, 
    autonomous vehicles, and facial recognition systems.
    """,
    """
    Natural language processing enables machines to understand human language. Modern NLP 
    uses transformer architectures like BERT and GPT. These models power chatbots, 
    translation services, and search engines. NLP is crucial for human-computer interaction.
    """,
    # Add more documents as needed
]
```

**Explanation**

**Task 6.1 – Creating an Extended Knowledge Base**

In this task, I created a more realistic dataset consisting of longer,
multi-sentence documents. Unlike earlier short definitions, these
documents contain multiple related ideas and practical examples.

This extended knowledge base better represents real-world RAG scenarios,
where documents are longer, more descriptive, and contain mixed content.
Handling such data requires effective chunking and embedding strategies
to preserve context while maintaining retrieval quality.

This dataset will allow further testing of scalability and retrieval
performance under more realistic conditions.

In [25]:
# Solution for Task 6.1

# Extending documents in dataset
EXTENDED_DOCS = [
    """
    Artificial Intelligence (AI) is transforming industries worldwide. Machine learning, 
    a subset of AI, enables systems to learn from data without explicit programming. 
    Companies use ML for predictive analytics, recommendation systems, and automation. 
    The field has grown exponentially with the availability of big data and computational power.
    """,

    """
    Deep learning has revolutionized computer vision and natural language processing. 
    Convolutional neural networks (CNNs) excel at image recognition tasks, achieving 
    human-level performance in many cases. Applications include medical imaging, 
    autonomous vehicles, and facial recognition systems.
    """,

    """
    Natural language processing enables machines to understand human language. Modern NLP 
    uses transformer architectures like BERT and GPT. These models power chatbots, 
    translation services, and search engines. NLP is crucial for human-computer interaction.
    """,

    """
    Reinforcement learning focuses on training agents through rewards and penalties. 
    It is widely used in robotics, game playing, and decision-making systems. 
    Algorithms like Q-learning and policy gradients help optimize sequential actions.
    """,

    """
    Computer vision allows machines to interpret visual data from images and videos. 
    Techniques such as object detection, image segmentation, and feature extraction 
    are widely applied in surveillance, healthcare, and augmented reality systems.
    """
]

In [26]:
# Testing Implementation

print("Number of extended documents:", len(EXTENDED_DOCS))

for i, doc in enumerate(EXTENDED_DOCS):
    print(f"\nDocument {i+1} preview:")
    print(doc.strip()[:150], "...")

Number of extended documents: 5

Document 1 preview:
Artificial Intelligence (AI) is transforming industries worldwide. Machine learning, 
    a subset of AI, enables systems to learn from data without e ...

Document 2 preview:
Deep learning has revolutionized computer vision and natural language processing. 
    Convolutional neural networks (CNNs) excel at image recognition ...

Document 3 preview:
Natural language processing enables machines to understand human language. Modern NLP 
    uses transformer architectures like BERT and GPT. These mod ...

Document 4 preview:
Reinforcement learning focuses on training agents through rewards and penalties. 
    It is widely used in robotics, game playing, and decision-making ...

Document 5 preview:
Computer vision allows machines to interpret visual data from images and videos. 
    Techniques such as object detection, image segmentation, and fea ...


**Observation**

The extended knowledge base contains 5 longer, multi-sentence documents,
each covering broader AI-related topics such as artificial intelligence,
deep learning, NLP, reinforcement learning, and computer vision.

Unlike the earlier short definition-based dataset, these documents contain
multiple related ideas and practical applications. This more closely
resembles real-world data, where documents are typically longer and more
context-rich.

This extended dataset will allow more realistic testing of chunking,
embedding generation, and retrieval performance in the RAG pipeline.

### Task 6.3: Build a Complete RAG Retrieval Demo

Create a simple interactive demo of your retrieval system.

```python
def interactive_retrieval_demo(pipeline):
    """
    Run an interactive demo of the retrieval system.
    """
    print("RAG Retrieval System Demo")
    print("=" * 50)
    print("Enter your queries (type 'quit' to exit)")
    print("=" * 50)
    
    # Your code here
    # Allow users to:
    # 1. Enter queries
    # 2. See retrieved documents with scores
    # 3. View statistics
    pass
```

**Explanation**

**Task 6.3 – Complete RAG Retrieval Demo (Chunk-Based)**

In this task, I implemented a production-style retrieval demo using
chunked documents instead of full-length documents.

Long documents are first split into smaller semantic chunks.
Each chunk is embedded separately to improve retrieval granularity.
This allows the system to return more precise and context-specific
results instead of entire long documents.

The interactive demo allows users to:
1. Enter queries
2. View retrieved chunks with similarity scores
3. See knowledge base statistics

This structure better reflects real-world RAG implementations.

In [27]:
# Solution to Task 6.3

# Step 1: Chunk extended documents
chunked_docs = []
for doc in EXTENDED_DOCS:
    chunks = chunk_by_sentences(doc.strip(), max_sentences=2)
    chunked_docs.extend(chunks)

print("\n=== Chunking Summary ===")
print(f"Total Chunks Created : {len(chunked_docs)}")
print("=" * 40)


# Step 2: Initialize pipeline and add chunked documents
extended_pipeline = RetrievalPipeline()
extended_pipeline.add_documents(chunked_docs)

stats = extended_pipeline.get_stats()

print("\n=== Extended Retrieval Pipeline Statistics ===")
print(f"{'Number of Documents':<22}: {stats['num_documents']}")
print(f"{'Embedding Dimension':<22}: {stats['embedding_dimension']}")
print("=" * 45)


# Step 3: Interactive Retrieval Demo
def interactive_retrieval_demo(pipeline):

    print("\nRAG Retrieval System Demo")
    print("=" * 50)
    print("Enter your queries (type 'quit' to exit)")
    print("=" * 50)

    while True:
        query = input("\nEnter Query: ").strip()

        if query.lower() == "quit":
            print("\nExiting demo. Thank you for using the RAG system.")
            break

        results = pipeline.retrieve(query, top_k=3, threshold=0.3)

        if not results:
            print("\nNo relevant results found above the similarity threshold.")
            continue

        print("\nTop Retrieved Chunks:")
        print("-" * 50)

        for i, (doc, score) in enumerate(results, 1):
            print(f"\nResult {i}")
            print(f"Similarity Score : {score:.4f}")
            print(f"Content:\n{doc.strip()}")
            print("-" * 50)

        stats = pipeline.get_stats()

        print("\nKnowledge Base Overview")
        print("-" * 40)
        print(f"{'Total Chunks':<18}: {stats['num_documents']}")
        print(f"{'Embedding Size':<18}: {stats['embedding_dimension']}")
        print("-" * 40)


=== Chunking Summary ===
Total Chunks Created : 9


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



=== Extended Retrieval Pipeline Statistics ===
Number of Documents   : 9
Embedding Dimension   : 384


In [28]:
# Implementation

# Run the demo
interactive_retrieval_demo(extended_pipeline)


RAG Retrieval System Demo
Enter your queries (type 'quit' to exit)



Enter Query:  what is machine learning?



Top Retrieved Chunks:
--------------------------------------------------

Result 1
Similarity Score : 0.7166
Content:
Artificial Intelligence (AI) is transforming industries worldwide. Machine learning, 
    a subset of AI, enables systems to learn from data without explicit programming.
--------------------------------------------------

Result 2
Similarity Score : 0.5101
Content:
Companies use ML for predictive analytics, recommendation systems, and automation. 
    The field has grown exponentially with the availability of big data and computational power.
--------------------------------------------------

Result 3
Similarity Score : 0.4305
Content:
Computer vision allows machines to interpret visual data from images and videos. 
    Techniques such as object detection, image segmentation, and feature extraction 
    are widely applied in surveillance, healthcare, and augmented reality systems.
--------------------------------------------------

Knowledge Base Overview
-----------


Enter Query:  applications of deep learning



Top Retrieved Chunks:
--------------------------------------------------

Result 1
Similarity Score : 0.5689
Content:
Applications include medical imaging, 
    autonomous vehicles, and facial recognition systems.
--------------------------------------------------

Result 2
Similarity Score : 0.5154
Content:
Deep learning has revolutionized computer vision and natural language processing. 
    Convolutional neural networks (CNNs) excel at image recognition tasks, achieving 
    human-level performance in many cases.
--------------------------------------------------

Result 3
Similarity Score : 0.4262
Content:
Computer vision allows machines to interpret visual data from images and videos. 
    Techniques such as object detection, image segmentation, and feature extraction 
    are widely applied in surveillance, healthcare, and augmented reality systems.
--------------------------------------------------

Knowledge Base Overview
----------------------------------------
Total Chunks  


Enter Query:  quit



Exiting demo. Thank you for using the RAG system.


**Observation**

**This is an informational warning related to internal transformer model
components and does not affect embedding generation or retrieval performance.
The model loads successfully and functions correctly despite this message.**

The extended knowledge base was successfully chunked into 9 smaller segments,
improving retrieval granularity compared to full-document embedding.

For the query "What is machine learning?", the top result correctly retrieved
the chunk explaining machine learning as a subset of AI, with a strong similarity
score (0.7166). The second result retrieved a related application-focused chunk,
demonstrating semantic awareness of context expansion.

For "Applications of deep learning", the system correctly retrieved a
chunk specifically listing deep learning applications (0.5689), followed
by a broader deep learning explanation (0.5154). This shows that chunking
allowed the system to return focused, context-specific results instead
of entire long documents.

**The model loading message showing "UNEXPECTED embeddings.position_ids"
is an informational warning from the transformer model and does not
affect retrieval functionality.**

Overall, chunk-based retrieval improves precision and relevance by returning
smaller, semantically aligned text segments. The interactive demo confirms
that the retrieval pipeline is functioning effectively in a realistic scenario.

## Challenge Tasks

### Challenge 1: Implement Re-ranking

After initial retrieval, re-rank results using a different strategy (e.g., keyword matching combined with semantic similarity).

**Explanation**

**Challenge 1 – Implementing Re-ranking with Hybrid Retrieval**

In this challenge, I enhanced the retrieval pipeline by introducing
a re-ranking strategy that combines semantic similarity with keyword
matching.

Initial retrieval is performed using cosine similarity between
embeddings. After obtaining the top-k results, the documents are
re-scored using a hybrid score:

Hybrid Score = (Semantic Score × weight) + (Keyword Match Score × weight)

Keyword matching helps emphasize documents that explicitly contain
important query terms, while semantic similarity captures contextual
meaning.

This hybrid approach improves precision by prioritizing documents
that are both semantically relevant and lexically aligned with the query.

In [29]:
# Solution to Challenge 1

def hybrid_retrieve(pipeline, query, top_k=3, threshold=0.0,
                    semantic_weight=0.7, keyword_weight=0.3):
    """
    Hybrid retrieval combining semantic similarity and keyword matching.
    
    Args:
        pipeline: RetrievalPipeline instance
        query: Query string
        top_k: Number of results
        threshold: Minimum semantic similarity
        semantic_weight: Weight for semantic similarity score
        keyword_weight: Weight for keyword matching score
    
    Returns:
        List of (document, hybrid_score) tuples
    """

    # Step 1: Perform normal semantic retrieval (without trimming top_k yet)
    semantic_results = pipeline.retrieve(query, top_k=len(pipeline.documents),
                                         threshold=threshold)

    query_tokens = set(query.lower().split())

    reranked_results = []

    for doc, semantic_score in semantic_results:
        # Step 2: Compute keyword match score
        doc_tokens = set(doc.lower().split())

        # Simple keyword overlap ratio
        keyword_matches = len(query_tokens & doc_tokens)
        keyword_score = keyword_matches / len(query_tokens) if query_tokens else 0

        # Step 3: Combine scores
        hybrid_score = (semantic_weight * semantic_score) + \
                       (keyword_weight * keyword_score)

        reranked_results.append((doc, hybrid_score))

    # Step 4: Sort by hybrid score
    reranked_results.sort(key=lambda x: x[1], reverse=True)

    return reranked_results[:top_k]

In [30]:
# Testing Implementation

query = "Applications of deep learning"

print("=== Hybrid Retrieval Results ===")
results = hybrid_retrieve(extended_pipeline, query, top_k=3)

for i, (doc, score) in enumerate(results, 1):
    print(f"\nResult {i}")
    print(f"Hybrid Score : {score:.4f}")
    print(doc.strip())

=== Hybrid Retrieval Results ===

Result 1
Hybrid Score : 0.5108
Deep learning has revolutionized computer vision and natural language processing. 
    Convolutional neural networks (CNNs) excel at image recognition tasks, achieving 
    human-level performance in many cases.

Result 2
Hybrid Score : 0.4732
Applications include medical imaging, 
    autonomous vehicles, and facial recognition systems.

Result 3
Hybrid Score : 0.3196
Artificial Intelligence (AI) is transforming industries worldwide. Machine learning, 
    a subset of AI, enables systems to learn from data without explicit programming.


**Observation – Hybrid Re-ranking**

And out of further curiosity, I experimented with hybrid re-ranking.After applying hybrid re-ranking, the order of retrieved results changed
compared to pure semantic retrieval.

The document providing a general explanation of deep learning is now ranked
first (Hybrid Score: 0.5108), while the applications-focused chunk moved to
second place (Hybrid Score: 0.4732).

This shift occurs because the keyword-based scoring favors documents that
contain stronger lexical overlap with the query terms ("deep" and "learning"),
not just semantic similarity.

Hybrid retrieval demonstrates how combining semantic and keyword signals
can influence ranking behavior. While semantic retrieval emphasizes meaning,
keyword matching boosts documents with explicit term overlap.

This approach provides more control over ranking and can improve precision
depending on the query type.

### Challenge 2: Add Metadata Filtering

Extend your pipeline to support filtering documents by metadata (e.g., date, category, author).

**Explantion**

Challenge 2 – Adding Metadata Filtering to the Retrieval Pipeline

In real-world RAG systems, documents often contain metadata such as
category, date, source, or author. Metadata filtering allows retrieval
to be restricted to documents that match specific criteria before
semantic similarity ranking is performed.

In this challenge, I extended the RetrievalPipeline class to:

- Store metadata for each document
- Support filtering documents based on metadata
- Perform retrieval only on filtered documents

This improves retrieval precision and allows domain-specific
search behavior, similar to production search engines.

In [31]:
# Solution for Challenge 2

class MetadataRetrievalPipeline(RetrievalPipeline):
    """
    Extended Retrieval Pipeline with Metadata Filtering.
    """

    def add_documents(self, documents, metadata_list=None):
        """
        Add documents with optional metadata.
        
        Args:
            documents (list): List of text documents
            metadata_list (list): List of metadata dictionaries (same length as documents)
        """

        if metadata_list and len(documents) != len(metadata_list):
            raise ValueError("metadata_list must match the number of documents.")

        self.documents = documents
        self.metadata = metadata_list if metadata_list else [{} for _ in documents]
        self.embeddings = create_embeddings(self.documents, self.model)


    def retrieve(self, query, top_k=3, threshold=0.0, metadata_filter=None):
        """
        Retrieve documents with optional metadata filtering.
        
        Args:
            query (str): Search query
            top_k (int): Number of results
            threshold (float): Minimum similarity score
            metadata_filter (dict): Filtering criteria (e.g., {"category": "NLP"})
        
        Returns:
            List of (document, score, metadata)
        """

        if self.embeddings is None:
            raise ValueError("No documents added.")

        query_embedding = self.model.encode(query, convert_to_numpy=True)

        results = []

        for i, doc_embedding in enumerate(self.embeddings):

            # Apply metadata filtering
            if metadata_filter:
                if not all(self.metadata[i].get(k) == v for k, v in metadata_filter.items()):
                    continue

            score = cosine_similarity(query_embedding, doc_embedding)

            if score >= threshold:
                results.append((self.documents[i], score, self.metadata[i]))

        results.sort(key=lambda x: x[1], reverse=True)

        return results[:top_k]

In [32]:
# Testing Implementation

# Create metadata for extended documents (before chunking, simple example)

metadata = [
    {"category": "AI", "author": "Research Team A"},
    {"category": "Deep Learning", "author": "Research Team B"},
    {"category": "NLP", "author": "Research Team C"},
    {"category": "Reinforcement Learning", "author": "Research Team D"},
    {"category": "Computer Vision", "author": "Research Team E"},
]

# Initialize metadata pipeline
meta_pipeline = MetadataRetrievalPipeline()

meta_pipeline.add_documents(EXTENDED_DOCS, metadata)

# Retrieve only NLP documents
query = "transformer models"
results = meta_pipeline.retrieve(
    query,
    top_k=3,
    metadata_filter={"category": "NLP"}
)

print("=== Metadata Filtered Retrieval Results ===")

for i, (doc, score, meta) in enumerate(results, 1):
    print(f"\nResult {i}")
    print(f"Score : {score:.4f}")
    print(f"Metadata : {meta}")
    print(doc.strip()[:200], "...")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


=== Metadata Filtered Retrieval Results ===

Result 1
Score : 0.2778
Metadata : {'category': 'NLP', 'author': 'Research Team C'}
Natural language processing enables machines to understand human language. Modern NLP 
    uses transformer architectures like BERT and GPT. These models power chatbots, 
    translation services, and ...


**Observation**
  
The metadata filtering functionality worked correctly. When applying
the filter {"category": "NLP"}, only documents labeled under the NLP
category were considered during retrieval.

The query "transformer models" returned the NLP-related document with
a similarity score of 0.2778. Although the similarity score is moderate,
the system correctly restricted the search to the specified metadata
category before performing semantic ranking.

**The model loading message showing "UNEXPECTED embeddings.position_ids"
is an informational warning from the transformer model and does not
affect retrieval functionality.**

The embedding model loaded successfully,
as confirmed by proper retrieval behavior.

Overall, metadata filtering successfully narrows the search scope,
demonstrating how real-world RAG systems combine semantic search with
structured filtering for improved precision.

### Challenge 3: Implement Hybrid Search

Combine dense retrieval (embeddings) with sparse retrieval (keyword-based like BM25) for improved results.

**Explanation**

**Challenge 3 – Implementing Hybrid Search (Dense + Sparse Retrieval)**

Hybrid search combines dense retrieval (semantic similarity using embeddings)
with sparse retrieval (keyword-based matching such as BM25).

Dense retrieval captures contextual meaning, while sparse retrieval emphasizes
exact keyword matches and term importance. Combining both approaches improves
retrieval robustness and ranking quality.

In this implementation:
- Dense score is computed using cosine similarity.
- Sparse score is computed using a simple BM25-style scoring mechanism
  based on term frequency and inverse document frequency.
- Final score = weighted combination of dense and sparse scores.

This hybrid approach is widely used in production search and RAG systems.

In [33]:
# Solution for Challenge 3

import math
from collections import Counter

def compute_idf(documents):
    """
    Compute Inverse Document Frequency (IDF) for all terms.
    """
    N = len(documents)
    idf = {}
    
    for doc in documents:
        tokens = set(doc.lower().split())
        for token in tokens:
            idf[token] = idf.get(token, 0) + 1

    for token in idf:
        idf[token] = math.log((N + 1) / (idf[token] + 1)) + 1

    return idf


def sparse_score(query, document, idf):
    """
    Compute simplified BM25-style sparse score.
    """
    query_tokens = query.lower().split()
    doc_tokens = document.lower().split()
    tf = Counter(doc_tokens)

    score = 0
    for term in query_tokens:
        if term in tf:
            score += tf[term] * idf.get(term, 0)

    return score


def hybrid_search(pipeline, query, top_k=3, 
                  dense_weight=0.6, sparse_weight=0.4):
    """
    Hybrid search combining dense and sparse retrieval.
    """

    documents = pipeline.documents
    idf = compute_idf(documents)

    query_embedding = pipeline.model.encode(query, convert_to_numpy=True)

    results = []

    for i, doc_embedding in enumerate(pipeline.embeddings):

        # Dense score
        dense = cosine_similarity(query_embedding, doc_embedding)

        # Sparse score
        sparse = sparse_score(query, documents[i], idf)

        # Normalize sparse score
        sparse = sparse / (len(query.split()) + 1)

        hybrid_score = (dense_weight * dense) + (sparse_weight * sparse)

        results.append((documents[i], hybrid_score, dense, sparse))

    results.sort(key=lambda x: x[1], reverse=True)

    return results[:top_k]

In [34]:
# Testing Implementation

query = "applications of deep learning"

print("=== Hybrid Search Results (Dense + Sparse) ===")

results = hybrid_search(extended_pipeline, query, top_k=3)

for i, (doc, hybrid_score, dense, sparse) in enumerate(results, 1):
    print(f"\nResult {i}")
    print(f"Hybrid Score : {hybrid_score:.4f}")
    print(f"Dense Score  : {dense:.4f}")
    print(f"Sparse Score : {sparse:.4f}")
    print(doc.strip()[:200], "...")

=== Hybrid Search Results (Dense + Sparse) ===

Result 1
Hybrid Score : 0.6943
Dense Score  : 0.5154
Sparse Score : 0.9627
Deep learning has revolutionized computer vision and natural language processing. 
    Convolutional neural networks (CNNs) excel at image recognition tasks, achieving 
    human-level performance in  ...

Result 2
Hybrid Score : 0.5501
Dense Score  : 0.5689
Sparse Score : 0.5219
Applications include medical imaging, 
    autonomous vehicles, and facial recognition systems. ...

Result 3
Hybrid Score : 0.3860
Dense Score  : 0.3495
Sparse Score : 0.4408
Artificial Intelligence (AI) is transforming industries worldwide. Machine learning, 
    a subset of AI, enables systems to learn from data without explicit programming. ...


**Observation**

The hybrid retrieval approach successfully combined dense semantic similarity
with sparse keyword-based scoring.

For the query "applications of deep learning", the top result achieved a
high hybrid score (0.6943), primarily due to a strong sparse score (0.9627).
This indicates significant keyword overlap between the query and the document.

Although the second result had a slightly higher dense score (0.5689),
its sparse score was lower, resulting in a slightly reduced hybrid ranking.
This demonstrates how keyword emphasis can influence ranking decisions
in hybrid systems.

The hybrid approach improves robustness by combining contextual understanding
(dense retrieval) with exact lexical matching (sparse retrieval). This mirrors
real-world production search systems where hybrid retrieval improves precision,
especially for keyword-sensitive queries.

## Questions

1. What are the key trade-offs between chunk size and retrieval quality?

2. How would you handle documents in multiple languages?

3. What strategies could you use to update your knowledge base without recomputing all embeddings?

4. How would you monitor retrieval quality in a production system?

5. What are the computational costs of your pipeline, and how would you optimize for large-scale deployment?

# Answers

**1. What are the key trade-offs between chunk size and retrieval quality?**

Chunk size directly affects retrieval precision and context preservation.

Smaller chunks:
- Improve precision and granularity.
- Allow the system to retrieve very specific information.
- May lose broader context if too small.

Larger chunks:
- Preserve more context and coherence.
- Reduce the number of embeddings.
- May decrease precision because irrelevant content is included with relevant text.

Therefore, an optimal chunk size balances contextual completeness and retrieval specificity.


**2. How would you handle documents in multiple languages?**

For multi-language documents, I would:

- Use multilingual embedding models (e.g., multilingual sentence transformers).
- Normalize text using language detection and preprocessing.
- Optionally store language metadata and filter by language.
- Ensure tokenization supports Unicode and non-English text.

This ensures embeddings capture semantic meaning across different languages.

    
**3. What strategies could you use to update your knowledge base without recomputing all embeddings?**

To avoid recomputing embeddings for the entire dataset:

- Only embed new or updated documents (incremental updates).
- Store embeddings in a vector database and append new vectors.
- Use versioning to track changes in documents.
- Apply batch updates instead of full recomputation.

This reduces computational cost and improves scalability.


**4. How would you monitor retrieval quality in a production system?**

In production, I would monitor:

- Precision and recall metrics on benchmark queries.
- Click-through rate or user interaction feedback.
- Query success rate (did users refine or repeat the query?).
- Latency and response time metrics.
- Drift detection to identify changes in document distribution.

Continuous evaluation ensures retrieval performance does not degrade over time.


**5. What are the computational costs of your pipeline, and how would you optimize for large-scale deployment?**

The main computational costs include:

- Embedding generation (model inference time).
- Similarity computation (especially for large document collections).
- Memory usage for storing high-dimensional vectors.

To optimize for large-scale deployment:

- Use vector databases (e.g., FAISS) for efficient nearest neighbor search.
- Apply approximate nearest neighbor (ANN) algorithms.
- Batch embedding generation.
- Use GPU acceleration.
- Reduce embedding dimensions if possible.
- Cache frequent queries.

These strategies ensure scalability when handling thousands or millions of documents.

**Took help from AI to give structured answers**

**Note:**

**Used AI for ensuring syntax correctness and to write detailed explanations and observations for the all the tasks and challenges .**

**Key Learnings from This Lab**

Through this lab, I understood how a Retrieval-Augmented Generation (RAG)
pipeline works step by step. I learned how document chunking affects
retrieval quality and why selecting an appropriate chunk size is important
for maintaining context while improving precision.

I gained hands-on experience in generating embeddings using sentence
transformer models and computing cosine similarity to measure semantic
relationships between documents and queries.

I implemented different retrieval strategies, including top-k retrieval,
threshold-based filtering, batch processing, and hybrid re-ranking. 
This helped me understand how ranking behavior changes depending on 
semantic and keyword signals.

I also learned how to evaluate retrieval systems using metrics such as
Precision@K and Recall@K, and how to analyze system performance.

Finally, through extended datasets, metadata filtering, and hybrid search,
I understood how real-world RAG systems are optimized for scalability,
efficiency, and practical deployment.